In [1]:
import os
os.environ["DOCLING_DISABLE_OCR"] = "1"
os.environ["RAPIDOCR_DISABLE"] = "1"


In [1]:
from langchain_docling.loader import DoclingLoader  

In [ ]:
from pathlib import Path

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.chunking import HybridChunker

from langchain_docling import DoclingLoader
from langchain_docling.loader import ExportType

PDF_PATH = Path(r"C:\Users\hhdor\OneDrive\Bureau\Ai4industrie\docs\DDRM_86.pdf")

# 1) Options PDF: désactiver OCR
pdf_opts = PdfPipelineOptions()
pdf_opts.do_ocr = False  # OCR OFF

# 2) Converter Docling configuré
converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pdf_opts)
    }
)

# 3) Chunker Docling (tokenizer = ton modèle d'embeddings)
EMBED_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
chunker = HybridChunker(
    tokenizer=EMBED_MODEL_ID,
    max_tokens=1000,
    overlap=100,
    )

# 4) Loader en mode DOC_CHUNKS (pas MARKDOWN)
loader = DoclingLoader(
    file_path=str(PDF_PATH),
    converter=converter,
    export_type=ExportType.DOC_CHUNKS,
    chunker=chunker,
)

docs = loader.load()

print("Nb doc-chunks:", len(docs))
print("Exemple metadata:", docs[0].metadata)
print("Extrait:", docs[0].page_content[:300])


Token indices sequence length is longer than the specified maximum sequence length for this model (541 > 512). Running this sequence through the model will result in indexing errors


Nb doc-chunks: 278
Exemple metadata: {'source': 'C:\\Users\\hhdor\\OneDrive\\Bureau\\Ai4industrie\\docs\\DDRM_86.pdf', 'dl_meta': {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/texts/5', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 3, 'bbox': {'l': 154.2, 't': 615.0440067282844, 'r': 549.847, 'b': 501.9680067282845, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 681]}]}, {'self_ref': '#/texts/6', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 3, 'bbox': {'l': 68.0, 't': 473.54400672828444, 'r': 552.033, 'b': 448.66800672828447, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 141]}]}, {'self_ref': '#/texts/7', 'parent': {'$ref': '#/groups/0'}, 'children': [], 'content_layer': 'body', 'label': 'list_item', 'prov': [{'page_no': 3, 'bbox': {'l': 86.0, 't': 435.7440067282845, 'r': 551.689, 'b': 385.66800672

In [5]:
from pathlib import Path

OUT_MD = Path("DDRM_86_docling_chunks.md")

with OUT_MD.open("w", encoding="utf-8") as f:
    for i, d in enumerate(docs, 1):
        meta = d.metadata
        f.write(f"\n\n---\n")
        f.write(f"### Chunk {i}\n")
        f.write(f"- page: {meta.get('page', 'N/A')}\n")
        f.write(f"- heading: {meta.get('heading', 'N/A')}\n\n")
        f.write(d.page_content)

print("Chunks écrits dans", OUT_MD)


Chunks écrits dans DDRM_86_docling_chunks.md


In [8]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embeddings)
vectorstore.save_local("faiss_ddrm_vienne_2024")

print("Index FAISS sauvegardé dans ./faiss_ddrm_vienne_2024")


Index FAISS sauvegardé dans ./faiss_ddrm_vienne_2024


In [9]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.load_local(
    "faiss_ddrm_vienne_2024",
    embeddings,
    allow_dangerous_deserialization=True
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 6})

print("Retriever prêt")


Retriever prêt


In [10]:
from langchain_core.tools.retriever import create_retriever_tool

search_tool = create_retriever_tool(
    retriever,
    name="search_ddrm",
    description=(
        "Recherche dans le DDRM de la Vienne (markdown nettoyé). "
        "Retourne des extraits pertinents avec leur section."
    )
)

print("Retriever tool prêt")


Retriever tool prêt


In [ ]:
from langchain_ollama import ChatOllama
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import InMemorySaver

SYSTEM_PROMPT = (
    "Tu es un assistant d'analyse documentaire (DDRM / risques). "
    "RÈGLE ABSOLUE : tu dois répondre UNIQUEMENT à partir du CONTEXTE "
    "fourni par l'outil search_ddrm. "
    "Si le contexte ne contient pas l'information, réponds exactement : "
    "'Information non trouvée dans les documents fournis.' "
    "Réponds en français, clair, structuré. "
    "Ajoute des citations au format "
    "[source: <source>, section: <section>]."
)

llm = ChatOllama(
    model="ministral-3",
    base_url="http://127.0.0.1:11434",
    temperature=0,
)

checkpointer = InMemorySaver()

agent = create_react_agent(
    model=llm,
    tools=[search_tool],
    prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
)

print("Agent prêt")


Agent prêt


C:\Users\hhdor\AppData\Local\Temp\ipykernel_23572\1888918847.py:24: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [ ]:
question = "Que faire en cas de glissement de terrains ?"

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Quels sont les principaux risques naturels ?"}]},
    config={"configurable": {"thread_id": "ddrm-session"}}
)

print(result["messages"][-1].content)


Dans le département de la Vienne, les **principaux risques naturels** identifiés sont les suivants :

1. **Inondations** : Risque majeur lié aux crues des cours d'eau, notamment dans certaines zones exposées.
   [source: Les risques naturels, section: LISTE DES COMMUNES PAR RISQUES NATURELS ET TECHNOLOGIQUE]

2. **Feux de forêt** : Risque lié à la propagation des incendies dans les zones forestières.
   [source: Les risques naturels, section: OÙ S'INFORMER SUR LE RISQUE INCENDIE]

3. **Mouvements de terrain** : Incluant les phénomènes liés aux cavités naturelles comme les **karsts, gouffres ou grottes**, ainsi que les **cavités de suffosion** (liées à l'érosion interne et aux circulations d'eau souterraine).
   [source: Les cavités naturelles, section: Les cavités naturelles]

4. **Tempêtes** : Bien que moins explicitement détaillées dans le contexte, elles sont mentionnées comme un risque naturel potentiel.
   [source: Les risques naturels, section: Pour ce qui concerne le département

In [17]:
for msg in result["messages"]:
    print(msg)

content='Quels sont les principaux risques naturels ?' additional_kwargs={} response_metadata={} id='cad243c4-6ca2-4bb2-907d-b0780db714bb'
content='' additional_kwargs={} response_metadata={'model': 'ministral-3', 'created_at': '2026-01-22T14:04:19.7782743Z', 'done': True, 'done_reason': 'stop', 'total_duration': 23058777700, 'load_duration': 20573632600, 'prompt_eval_count': 194, 'prompt_eval_duration': 678490200, 'eval_count': 16, 'eval_duration': 1677767000, 'logprobs': None, 'model_name': 'ministral-3', 'model_provider': 'ollama'} id='lc_run--019be604-db92-78d0-b2e2-7791f74d47df-0' tool_calls=[{'name': 'search_ddrm', 'args': {'query': 'risques naturels Vienne'}, 'id': '51fa5ea1-fe92-4599-b16d-587071463ac8', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 194, 'output_tokens': 16, 'total_tokens': 210}
content="3 - QUELS SONT LES RISQUES DANS LE DÉPARTEMENT ?\nLe département de la Vienne compte peu de risques majeurs.\n\nOÙ S'INFORMER  SUR LE RISQUE INONDA

In [9]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vs = FAISS.load_local(
    "faiss_ddrm_vienne_2024",
    embeddings,
    allow_dangerous_deserialization=True
)

print("Nb docs dans l'index:", len(vs.docstore._dict))
print("Dimension embedding (ex):", len(embeddings.embed_query("test")))


Nb docs dans l'index: 232
Dimension embedding (ex): 384


In [12]:
query = "Que faire en cas de glissement de terrains ?"
results = vs.similarity_search_with_score(query, k=5)

for i, (doc, score) in enumerate(results, 1):
    print("\n---", i, "score:", score)
    print("meta:", doc.metadata)
    print(doc.page_content[:600])



--- 1 score: 0.5993333
meta: {'source': 'DDRM_Vienne_2024.md', 'section': '\uf0a7 Le glissement de terrain   :'}
##  Le glissement de terrain   :
Il se produit généralement en situation de forte saturation des sols en eau. Il correspond au déplacement de terrains meubles ou rocheux le long d'une surface de rupture. Les volumes de terrain en mouvement peuvent être considérables.

--- 2 score: 0.6082244
meta: {'source': 'DDRM_Vienne_2024.md', 'section': 'LES MESURES DE PRÉVENTION'}
-  La construction adaptée :   la   diversité   des   phénomènes   de   mouvements de terrain implique que des mesures très spécifiques soient mises en œuvre à titre individuel. La protection contre le retrait-gonflement des argiles nécessite des mesures relativement simples d'adaptation du bâtiment au contexte local ;
-  La consultation de la base de données avant tout aménagement : sur les mouvements de terrain, la présence de cavités souterraines et sur l'aléa retrait-gonflement des argiles : https://ww

In [ ]:
def run_query(agent_executor, query, thread_id="thread-fr-1"):
    print("\n" + "=" * 60)
    print(f"❓ Question: {query}")
    print("=" * 60)

    config_dict = {"configurable": {"thread_id": thread_id}}

    try:
        response = agent_executor.invoke(
            {"messages": [{"role": "user", "content": query}]},
            config=config_dict,
        )

        # Affichage réponse
        print("\n🧠 Réponse de l'agent :")
        if response.get("messages"):
            print(response["messages"][-1].content)
        else:
            print("⚠️ Pas de réponse générée.")

        # Vérification de l'usage de l'outil
        print("\n🔍 --- Vérification de l'utilisation de l'outil ---")
        has_used_tool = False
        for msg in response.get("messages", []):
            if msg.type == "tool":
                print("✅ L'agent a bien consulté le DDRM.")
                has_used_tool = True
                break

        if not has_used_tool:
            print("❌ ATTENTION : L'agent a répondu sans lire le document !")

    except Exception as e:
        print(f"❌ Erreur lors de l'exécution : {e}")

questions = [
    "Quels sont les risques naturels et technologiques majeurs recensés dans le département de la Vienne selon le DDRM ?",
    "Quelles sont les principales rivières concernées par le risque inondation dans la Vienne et quels types d'inondations peuvent survenir ?",
    "Comment est classé le département de la Vienne en termes de zonage sismique ?",
    "Quelles sont les mesures de pré-distribution de comprimés d'iode prévues autour de la centrale nucléaire de Civaux ?",
    "Quels sont les établissements classés SEVESO seuil haut dans le département de la Vienne et où sont-ils situés ?",
    "De quoi se compose le Signal National d'Alerte et comment reconnaît-on le signal de fin d'alerte ?"
]

for q in questions:
    run_query(agent, q)



❓ Question: Quels sont les risques naturels et technologiques majeurs recensés dans le département de la Vienne selon le DDRM ?
